In [49]:
import os
import torch
from d2l import torch as d2l

# Raw bilingual text
# → Clean text
# → Source/Target 분리
# → Word-Level Tokenization
# → <eos> 추가
# -> Sequence Length 히스토그램 확인

In [50]:
# Preprocessing & Tokenization

class MTFraEng(d2l.DataModule):
    
    # 1) English-French Dataset Download
    def _download(self) -> str:
        archive_path = d2l.download(
            d2l.DATA_URL + "fra-eng.zip",
            self.root,
            "94646ad1522d915e7b0f9296181140edcf86a4f5",
        )

        d2l.extract(
            archive_path
        )

        text_path = os.path.join(
            self.root,
            "fra-eng",
            "fra.txt",
        )

        with open(
            text_path,
            encoding="utf-8",
        ) as file:
            return file.read()


    # 2) Text Preprocessing
    def _preprocess(
        self,
        text: str,
    ) -> str:

        # 1) Unicode Space -> 일반 Space    
        text = (
            text
            .replace("\u202f", " ")
            .replace("\xa0", " ")
        )

        # 2) Uppercase -> Lowercase
        lower_text = text.lower()
        output: list[str] = []
        
        for index, char in enumerate(
            lower_text
        ):
            is_punctuation = (
                char in ",.!?"
            )
            
            has_previous_character = (
                index > 0
            )
            
            previous_is_not_space = (
                has_previous_character
                and lower_text[index - 1] != " "
            )

            # 3) Punctuation 앞에 Space 삽입
            # : "go." → "go ."
            if (
                is_punctuation
                and previous_is_not_space
            ):
                output.append(" ")

            output.append(
                char
            )

        return "".join(output)


    # 3) Word-Level Tokenization
    def _tokenize(
        self,
        text: str,
        max_examples: int | None = None,
    ) -> tuple[
        list[list[str]],
        list[list[str]],
    ]:
        
        source: list[list[str]] = []
        target: list[list[str]] = []
        
        # 전체 text를 newline 기준으로 분리
        # : "go .\tva !"
        for line in text.splitlines():
            
            if (
                max_examples is not None
                and len(source) >= max_examples
            ):
                break
            
            
            # Tab을 기준으로 English와 French를 분리:
            parts = line.split("\t")
            
            
            # 정상적인 Translation pair가 아니라면 pass
            if len(parts) != 2:
                continue
        
        
            # part[0]: "go ."
            # part[1]: "va !"
            source_text = parts[0]
            target_text = parts[1]
            
            source_tokens = source_text.split() # source: ["go", "."]
            target_tokens = target_text.split() # target: ["va", "!"]
            
            source_tokens.append("<eos>") # ["go", ".", "<eos>"]
            target_tokens.append("<eos>") # ["va", "!", "<eos>"]
            
            source.append(
                source_tokens
            )
            target.append(
                target_tokens
            )
            
        return source, target

In [51]:
# Dataset Initialization

def mtfraeng_init(
    self: MTFraEng,
    batch_size: int,
    num_steps: int = 9,
    num_train: int = 512,
    num_val: int = 128,
    root: str = "../data",
    num_workers: int = 4,
) -> None:
    
    # Initialize Dataset root & workers
    d2l.DataModule.__init__(
        self,
        root=root,
        num_workers=num_workers,
    )
    
    self.batch_size = batch_size
    self.num_steps = num_steps
    self.num_train = num_train
    self.num_val = num_val
    
    (
        self.arrays,
        self.src_vocab,
        self.tgt_vocab,
    ) = self._build_arrays(
        self._download()
    )
    
setattr(
    MTFraEng,
    "__init__",
    mtfraeng_init,
)

In [52]:
# Padding & Truncation

def pad_or_trim(
    sequence: list[str],
    length: int,
) -> list[str]:
    
    if len(sequence) > length:
        return sequence[:length]
    
    return sequence + (
        ["<pad>"]
        * (length - len(sequence))
    )
    
    
short_sequence = [
    "go",
    ".",
    "<eos>",
]

long_sequence = [
    "i",
    "really",
    "like",
    "this",
    "book",
    ".",
    "<eos>",
]

print(
    "Padded:",
    pad_or_trim(
        short_sequence,
        length=5,
    ),
)

print(
    "Trimmed:",
    pad_or_trim(
        long_sequence,
        length=5,
    ),
)

Padded: ['go', '.', '<eos>', '<pad>', '<pad>']
Trimmed: ['i', 'really', 'like', 'this', 'book']


In [53]:
# Fixed-Length array construction

def mtfraeng_build_arrays(
    self: MTFraEng,
    raw_text: str,
    src_vocab: d2l.Vocab | None = None,
    tgt_vocab: d2l.Vocab | None = None,
) -> tuple[
    tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
    ],
    d2l.Vocab,
    d2l.Vocab,
]:
    
    
    def build_array(
        sentences: list[list[str]],
        vocab: d2l.Vocab | None,
        is_target: bool = False,
    ) -> tuple[
        torch.Tensor,
        d2l.Vocab,
        torch.Tensor,
    ]:
        
        # 모든 Sequence를 고정된 길이 "num_steps" 로 맞춤
        normalized_sentences = [
            pad_or_trim(
                sentence,
                self.num_steps,
            )
            for sentence in sentences
        ]
        
        # <bos> 추가 후 길이: num_steps + 1
        if is_target:
            normalized_sentences = [
                ["<bos>"] + sentence
                for sentence
                in normalized_sentences
            ]
            
        if vocab is None:
            vocab = d2l.Vocab(
                normalized_sentences,
                min_freq=2,
            )
         
        # ["go", ".", "<eos>", "<pad>"]
        #             ↓ Vocabulary
        #      [47, 12, 3, 4]   
        array = torch.tensor(
            [
                vocab[sentence]
                for sentence
                in normalized_sentences
            ],
            dtype=torch.long,
        )
        
        # ["go", ".", "<eos>", "<pad>", "<pad>"]
        #  -> 3
        valid_len = (
            array != vocab["<pad>"]
        ).to(torch.int32).sum(
            dim=1
        )
        
        return (
            array,
            vocab,
            valid_len,
        )
        
        
    source, target = self._tokenize(
        self._preprocess(raw_text),
        self.num_train + self.num_val,
    )
    
    # Source array
    (
        source_array,
        src_vocab,
        source_valid_len,
    ) = build_array(
        source,
        src_vocab,
    )

    # Target array:
    #  ["<bos>", "va", "!", "<eos>", "<pad>", ...]
    (
        target_array,
        tgt_vocab,
        _,
    ) = build_array(
        target,
        tgt_vocab,
        is_target=True,
    )
    
    # 마지막 Token 제거
    # ["<bos>", "va", "!", "<eos>", ...]
    decoder_input = target_array[
        :,
        :-1,
    ]
    
    # Label은 첫 Token인 <bos> 제거
    # ["va", "!", "<eos>", "<pad>", ...]
    label = target_array[
        :,
        1:,
    ]

    arrays = (
        source_array,
        decoder_input,
        source_valid_len,
        label,
    )

    return (
        arrays,
        src_vocab,
        tgt_vocab,
    )


setattr(
    MTFraEng,
    "_build_arrays",
    mtfraeng_build_arrays,
)

In [54]:
 # Fixed-Length Tensor Verification

data = MTFraEng(
    batch_size=3
)

(
    source_array,     # Encoder가 읽는 English 문장
    decoder_input,    # Decoder에 입력하는 이전 French token
    source_valid_len, # English 문장의 실제 길이
    label,            # Decoder가 예측해야 하는 다음 French Token
) = data.arrays

print(
    "Source shape:",
    tuple(source_array.shape),
)

print(
    "Decoder input shape:",
    tuple(decoder_input.shape),
)

print(
    "Source valid length shape:",
    tuple(source_valid_len.shape),
)

print(
    "Label shape:",
    tuple(label.shape),
)

print(
    "\nSource tokens:",
    data.src_vocab.to_tokens(
        source_array[0].to(torch.int32)
    ),
)

print(
    "Decoder input tokens:",
    data.tgt_vocab.to_tokens(
        decoder_input[0].to(torch.int32)
    ),
)

print(
    "Label tokens:",
    data.tgt_vocab.to_tokens(
        label[0].to(torch.int32)
    ),
)

print(
    "\nDecoder shift matches:",
    torch.equal(
        decoder_input[:, 1:],
        label[:, :-1],
    ),
)

Source shape: (640, 9)
Decoder input shape: (640, 9)
Source valid length shape: (640,)
Label shape: (640, 9)

Source tokens: ['go', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Decoder input tokens: ['<bos>', 'va', '!', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Label tokens: ['va', '!', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

Decoder shift matches: True


In [55]:
# DataLoader

def mtfraeng_get_dataloader(
    self: MTFraEng,
    train: bool,
) -> torch.utils.data.DataLoader:
    
    indices = (
        slice(0, self.num_train)
        if train
        else slice(self.num_train, None)
    )
    
    return self.get_tensorloader(
        self.arrays,
        train,
        indices,
    )

    
setattr(
    MTFraEng,
    "get_dataloader",
    mtfraeng_get_dataloader,
)

data = MTFraEng(
    batch_size=3,
)

(
    source,
    decoder_input,
    source_valid_len,
    label,
) = next(
    iter(data.train_dataloader())
)

print(
    "Source:\n",
    source.to(torch.int32),
)
print(
    "\nDecoder input:\n",
    decoder_input.to(torch.int32),
)
print(
    "\nSource length excluding <pad>:\n",
    source_valid_len.to(torch.int32),
)
print(
    "\nLabel:\n",
    label.to(torch.int32),
)

Source:
 tensor([[ 71,   1,   5,   2,   3,   4,   4,   4,   4],
        [ 16, 140,   2,   3,   4,   4,   4,   4,   4],
        [193, 105,   2,   3,   4,   4,   4,   4,   4]], dtype=torch.int32)

Decoder input:
 tensor([[  3, 175,   1, 117,   6,   0,   4,   5,   5],
        [  3,  18,   0,   4,   5,   5,   5,   5,   5],
        [  3, 206,   6, 147,   2,   4,   5,   5,   5]], dtype=torch.int32)

Source length excluding <pad>:
 tensor([5, 4, 4], dtype=torch.int32)

Label:
 tensor([[175,   1, 117,   6,   0,   4,   5,   5,   5],
        [ 18,   0,   4,   5,   5,   5,   5,   5,   5],
        [206,   6, 147,   2,   4,   5,   5,   5,   5]], dtype=torch.int32)


In [56]:
# 5) User-Provided Sentence Pair Conversion

def mtfraeng_build(
    self: MTFraEng,
    source_sentences: list[str],
    target_sentences: list[str],
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    raw_text = "\n".join(
        source + "\t" + target
        for source, target in zip(
            source_sentences,
            target_sentences,
        )
    )

    arrays, _, _ = self._build_arrays(
        raw_text,
        self.src_vocab,
        self.tgt_vocab,
    )

    return arrays


setattr(
    MTFraEng,
    "build",
    mtfraeng_build,
)


source, decoder_input, _, _ = mtfraeng_build(
    data,
    ["hi ."],
    ["salut ."],
)

print(
    "Source:",
    data.src_vocab.to_tokens(
        source[0].to(torch.int32)
    ),
)
print(
    "Target:",
    data.tgt_vocab.to_tokens(
        decoder_input[0].to(torch.int32)
    ),
)

Source: ['hi', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Target: ['<bos>', 'salut', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
